# CSCI 677 Homework 4 - Fine-tuning Faster R-CNN and DINO on Pascal VOC

**Name:** _replace with your name_  
**Course:** CSCI 677, Fall 2026  
**Models:** Detectron2 Faster R-CNN R101-FPN 3x and IDEA-Research DINO-4scale

This notebook is the submission and experiment log. Run it from top to bottom in a local Jupyter environment with a CUDA-capable GPU. It downloads Pascal VOC 2012, fine-tunes both detectors, reports AP50, saves qualitative detections, plots Detectron2 loss curves, and builds a data-backed comparison. Training outputs are stored in this repository under `outputs/`.

> Do not add made-up results. After the two training sections finish, run the final summary cells and export this executed notebook to PDF.

## 1. Reproducibility and experiment settings

The learning rates are deliberately below the common from-scratch settings. The default assignment-sized run uses 3,000 Faster R-CNN iterations and one DINO epoch. Increase `DINO_EPOCHS` to 2 only if runtime permits.

In [4]:
from pathlib import Path
import json, os, random, shutil, subprocess, sys, time

SEED = 677
FRCNN_MAX_ITER = 3000
FRCNN_LR = 2.5e-4
DINO_EPOCHS = 1
DINO_LR = 1e-5
DINO_BACKBONE_LR = 1e-6
SCORE_THRESHOLD = 0.50

random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
except ImportError:
    pass

print({
    'seed': SEED, 'frcnn_max_iter': FRCNN_MAX_ITER, 'frcnn_lr': FRCNN_LR,
    'dino_epochs': DINO_EPOCHS, 'dino_lr': DINO_LR,
    'dino_backbone_lr': DINO_BACKBONE_LR
})

{'seed': 677, 'frcnn_max_iter': 3000, 'frcnn_lr': 0.00025, 'dino_epochs': 1, 'dino_lr': 1e-05, 'dino_backbone_lr': 1e-06}


In [5]:
# Local runtime setup. All generated files remain inside this repository.
PROJECT = Path.cwd().resolve()
OUTPUT_ROOT = PROJECT / 'outputs'

VOC_ROOT = PROJECT / 'datasets' / 'VOC2012'
COCO_ROOT = PROJECT / 'voc2012_coco'
FRCNN_OUT = OUTPUT_ROOT / 'faster_rcnn_r101_fpn'
DINO_OUT = OUTPUT_ROOT / 'dino_4scale'
for directory in (OUTPUT_ROOT, FRCNN_OUT, DINO_OUT):
    directory.mkdir(parents=True, exist_ok=True)

try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
except ModuleNotFoundError:
    CUDA_AVAILABLE = False

if CUDA_AVAILABLE:
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: CUDA is not currently available. Setup and data preparation can run, but model training requires a CUDA-capable local environment.')
print('Runtime: local VS Code/Jupyter')
print('Outputs:', OUTPUT_ROOT)

Runtime: local VS Code/Jupyter
Outputs: C:\Users\richard\Documents\Faster-Rcnn-Dino-Finetuning\outputs


## 2. Install Detectron2 and DINO

DINO contains a custom CUDA operator, so this setup cell compiles it for the active local CUDA/PyTorch runtime. If a later import fails after installation, restart the kernel once and resume here.

In [6]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja', 'gdown', 'pycocotools'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', 'git+https://github.com/facebookresearch/detectron2.git'], check=True)
if not (PROJECT / 'DINO').exists():
    subprocess.run(['git', 'clone', 'https://github.com/IDEA-Research/DINO.git', str(PROJECT / 'DINO')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'DINO' / 'requirements.txt')], check=True)
# DINO predates PyTorch 2.6, whose torch.load default changed to weights_only=True.
# Its trusted official checkpoints include metadata, so make the legacy intent explicit.
dino_main_path = PROJECT / 'DINO' / 'main.py'
dino_main_source = dino_main_path.read_text()
if 'weights_only=False' not in dino_main_source:
    dino_main_source = dino_main_source.replace("map_location='cpu')", "map_location='cpu', weights_only=False)")
    dino_main_path.write_text(dino_main_source)
subprocess.run([sys.executable, 'setup.py', 'build', 'install'], cwd=PROJECT / 'DINO' / 'models' / 'dino' / 'ops', check=True)
subprocess.run([sys.executable, 'test.py'], cwd=PROJECT / 'DINO' / 'models' / 'dino' / 'ops', check=True)

CalledProcessError: Command '['c:\\Users\\richard\\Documents\\Faster-Rcnn-Dino-Finetuning\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-q', '--no-build-isolation', 'git+https://github.com/facebookresearch/detectron2.git']' returned non-zero exit status 2.

## 3. Download Pascal VOC 2012

The official VOC train/validation archive is unpacked into the directory layout expected by Detectron2.

In [ ]:
archive = PROJECT / 'VOCtrainval_11-May-2012.tar'
if not VOC_ROOT.exists():
    if not archive.exists():
        subprocess.run(['wget', '-q', 'http://data.brainchip.com/dataset-mirror/voc/VOCtrainval_11-May-2012.tar', '-O', str(archive)], check=True)
    subprocess.run(['tar', '-xf', str(archive), '-C', str(PROJECT)], check=True)
    (PROJECT / 'datasets').mkdir(exist_ok=True)
    shutil.move(str(PROJECT / 'VOCdevkit' / 'VOC2012'), str(VOC_ROOT))
    shutil.rmtree(PROJECT / 'VOCdevkit', ignore_errors=True)

required = [VOC_ROOT / 'Annotations', VOC_ROOT / 'JPEGImages', VOC_ROOT / 'ImageSets' / 'Main']
assert all(path.exists() for path in required), f'Incomplete VOC extraction under {VOC_ROOT}'
for split in ('train', 'val'):
    ids = (VOC_ROOT / 'ImageSets' / 'Main' / f'{split}.txt').read_text().split()
    print(split, len(ids), 'images')

## 4. Faster R-CNN R101-FPN fine-tuning

The COCO-pretrained R101-FPN 3x checkpoint initializes the detector. The classification head is replaced for the 20 Pascal VOC classes. Validation uses Detectron2's Pascal VOC evaluator, whose `AP50` is the assignment's requested metric.

In [ ]:
os.environ['DETECTRON2_DATASETS'] = str(PROJECT / 'datasets')
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import PascalVOCDetectionEvaluator

class VOCTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        return PascalVOCDetectionEvaluator(dataset_name)

cfg = get_cfg()
config_name = 'COCO-Detection/faster_rcnn_R_101_FPN_3x.yaml'
cfg.merge_from_file(model_zoo.get_config_file(config_name))
cfg.OUTPUT_DIR = str(FRCNN_OUT)
cfg.DATASETS.TRAIN = ('voc_2012_train',)
cfg.DATASETS.TEST = ('voc_2012_val',)
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(config_name)
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 20
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESHOLD
cfg.SOLVER.IMS_PER_BATCH = 1
cfg.SOLVER.BASE_LR = FRCNN_LR
cfg.SOLVER.MAX_ITER = FRCNN_MAX_ITER
cfg.SOLVER.STEPS = []
cfg.SOLVER.CHECKPOINT_PERIOD = 500
cfg.TEST.EVAL_PERIOD = 0  # evaluate once after training to save time
cfg.SEED = SEED

trainer = VOCTrainer(cfg)
trainer.resume_or_load(resume=(FRCNN_OUT / 'last_checkpoint').exists())
trainer.train()

In [ ]:
# Quantitative evaluation: Pascal VOC AP50 on the held-out VOC 2012 validation split.
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import build_detection_test_loader
from detectron2.evaluation import inference_on_dataset
from detectron2.modeling import build_model

cfg.MODEL.WEIGHTS = str(FRCNN_OUT / 'model_final.pth')
model = build_model(cfg)
DetectionCheckpointer(model).load(cfg.MODEL.WEIGHTS)
frcnn_results = inference_on_dataset(
    model,
    build_detection_test_loader(cfg, 'voc_2012_val'),
    PascalVOCDetectionEvaluator('voc_2012_val'),
)
def jsonable(value):
    if isinstance(value, dict): return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [jsonable(v) for v in value]
    if hasattr(value, 'item'): return value.item()
    return value

(FRCNN_OUT / 'evaluation.json').write_text(json.dumps(jsonable(frcnn_results), indent=2))
print(json.dumps(jsonable(frcnn_results), indent=2))

In [ ]:
# Plot the evolution of all available Detectron2 loss terms.
import pandas as pd
import matplotlib.pyplot as plt

records = [json.loads(line) for line in (FRCNN_OUT / 'metrics.json').read_text().splitlines()]
history = pd.DataFrame(records).sort_values('iteration')
loss_columns = [c for c in ['total_loss', 'loss_cls', 'loss_box_reg', 'loss_rpn_cls', 'loss_rpn_loc'] if c in history]
assert loss_columns, 'No loss columns found in metrics.json'

fig, axes = plt.subplots(2, 1, figsize=(11, 9), sharex=True)
for column in loss_columns:
    series = history.set_index('iteration')[column].dropna()
    axes[0].plot(series.index, series, alpha=0.18)
    axes[0].plot(series.index, series.rolling(50, min_periods=1).mean(), label=column)
axes[0].set_title('Faster R-CNN training losses (50-iteration moving average)')
axes[0].set_ylabel('Loss')
axes[0].legend(ncol=2)
axes[0].grid(alpha=0.25)
if 'lr' in history:
    axes[1].plot(history['iteration'], history['lr'])
axes[1].set_title('Learning rate')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('LR')
axes[1].grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FRCNN_OUT / 'training_curves.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# Qualitative results on a reproducible sample of validation images.
import cv2
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer

predictor = DefaultPredictor(cfg)
val_records = DatasetCatalog.get('voc_2012_val')
sample_records = random.Random(SEED).sample(val_records, 4)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, record in zip(axes.ravel(), sample_records):
    image_bgr = cv2.imread(record['file_name'])
    outputs = predictor(image_bgr)
    rendered = Visualizer(
        image_bgr[:, :, ::-1],
        metadata=MetadataCatalog.get('voc_2012_val'),
        scale=0.8,
    ).draw_instance_predictions(outputs['instances'].to('cpu')).get_image()
    ax.imshow(rendered)
    ax.set_title(Path(record['file_name']).name)
    ax.axis('off')
fig.suptitle('Faster R-CNN detections on VOC 2012 validation images', fontsize=16)
fig.tight_layout()
fig.savefig(FRCNN_OUT / 'qualitative_detections.png', dpi=180, bbox_inches='tight')
plt.show()

## 5. Convert Pascal VOC XML annotations to COCO JSON for DINO

DINO's COCO loader uses category IDs directly as class indices. Therefore this conversion uses contiguous IDs **0-19**, sets `num_classes=20`, and sets `dn_labelbook_size=21`. VOC boxes are inclusive and one-based; the converter changes them to zero-based COCO `[x, y, width, height]` boxes. Difficult objects are marked as crowd annotations so COCO evaluation ignores them.

In [ ]:
import xml.etree.ElementTree as ET
from PIL import Image

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat',
    'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]
CLASS_TO_ID = {name: idx for idx, name in enumerate(VOC_CLASSES)}

def convert_voc_split(split):
    split_ids = (VOC_ROOT / 'ImageSets' / 'Main' / f'{split}.txt').read_text().split()
    image_dir = COCO_ROOT / ('train2017' if split == 'train' else 'val2017')
    image_dir.mkdir(parents=True, exist_ok=True)
    images, annotations = [], []
    annotation_id = 1
    for image_id, stem in enumerate(split_ids, start=1):
        xml_root = ET.parse(VOC_ROOT / 'Annotations' / f'{stem}.xml').getroot()
        size = xml_root.find('size')
        width, height = int(size.findtext('width')), int(size.findtext('height'))
        source_image = VOC_ROOT / 'JPEGImages' / f'{stem}.jpg'
        target_image = image_dir / source_image.name
        if not target_image.exists():
            os.symlink(source_image, target_image)
        images.append({'id': image_id, 'file_name': source_image.name, 'width': width, 'height': height})
        for obj in xml_root.findall('object'):
            name = obj.findtext('name')
            if name not in CLASS_TO_ID:
                raise ValueError(f'Unknown VOC class: {name}')
            box = obj.find('bndbox')
            xmin, ymin = int(float(box.findtext('xmin'))), int(float(box.findtext('ymin')))
            xmax, ymax = int(float(box.findtext('xmax'))), int(float(box.findtext('ymax')))
            x, y = xmin - 1, ymin - 1
            box_width, box_height = xmax - xmin + 1, ymax - ymin + 1
            if not (0 <= x < width and 0 <= y < height and box_width > 0 and box_height > 0):
                raise ValueError(f'Invalid box in {stem}: {(xmin, ymin, xmax, ymax)}')
            annotations.append({
                'id': annotation_id, 'image_id': image_id, 'category_id': CLASS_TO_ID[name],
                'bbox': [x, y, box_width, box_height], 'area': box_width * box_height,
                'iscrowd': int(obj.findtext('difficult', default='0')),
            })
            annotation_id += 1
    payload = {
        'info': {'description': f'Pascal VOC 2012 {split} converted for DINO'},
        'licenses': [], 'images': images, 'annotations': annotations,
        'categories': [{'id': idx, 'name': name, 'supercategory': 'object'} for idx, name in enumerate(VOC_CLASSES)],
    }
    destination = COCO_ROOT / 'annotations' / f'instances_{split}2017.json'
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(payload))
    return destination, len(images), len(annotations)

for split in ('train', 'val'):
    print(split, convert_voc_split(split))

In [ ]:
# Validate both JSON files before launching an expensive training job.
from pycocotools.coco import COCO
for split in ('train', 'val'):
    json_path = COCO_ROOT / 'annotations' / f'instances_{split}2017.json'
    coco = COCO(str(json_path))
    assert sorted(coco.getCatIds()) == list(range(20))
    assert len(coco.getImgIds()) > 0 and len(coco.getAnnIds()) > 0
    print(split, 'images=', len(coco.getImgIds()), 'annotations=', len(coco.getAnnIds()))

## 6. DINO-4scale fine-tuning

Download `checkpoint0011_4scale.pth` from the assignment's official model-zoo folder and place it in `MyDrive/CSCI677_HW4_outputs/`. The assertion below prevents accidentally training from scratch. The classification-related weights are ignored because the pretrained checkpoint has a different class space.

In [ ]:
# Official model-zoo folder: https://drive.google.com/drive/folders/1qD5m1NmK0kjE5hh-G17XUX751WsEG-h_
DINO_PRETRAINED = OUTPUT_ROOT / 'checkpoint0011_4scale.pth'
assert DINO_PRETRAINED.exists(), (
    f'Download checkpoint0011_4scale.pth from the official DINO model-zoo folder and place it at {DINO_PRETRAINED}'
)
print('Using pretrained checkpoint:', DINO_PRETRAINED, f'({DINO_PRETRAINED.stat().st_size / 1e6:.1f} MB)')

In [ ]:
# Single-GPU fine-tuning. With one epoch, the built-in end-of-epoch evaluation is also the final evaluation.
dino_command = [
    sys.executable, 'main.py',
    '-c', 'config/DINO/DINO_4scale.py',
    '--coco_path', str(COCO_ROOT),
    '--output_dir', str(DINO_OUT),
    '--pretrain_model_path', str(DINO_PRETRAINED),
    '--finetune_ignore', 'label_enc.weight', 'class_embed',
    '--num_workers', '2', '--amp',
    '--options',
    f'epochs={DINO_EPOCHS}', f'lr={DINO_LR}', f'lr_backbone={DINO_BACKBONE_LR}',
    'num_classes=20', 'dn_labelbook_size=21', 'batch_size=1',
    'use_ema=False', 'save_checkpoint_interval=1',
]
print(' '.join(map(str, dino_command)))
subprocess.run(dino_command, cwd=PROJECT / 'DINO', check=True)

In [ ]:
# Parse DINO's final COCO evaluation vector: [AP, AP50, AP75, AP_small, AP_medium, AP_large, ...].
dino_log_rows = [json.loads(line) for line in (DINO_OUT / 'log.txt').read_text().splitlines() if line.strip()]
dino_eval_rows = [row for row in dino_log_rows if 'test_coco_eval_bbox' in row]
assert dino_eval_rows, 'DINO did not write COCO evaluation results.'
dino_eval = dino_eval_rows[-1]['test_coco_eval_bbox']
dino_metrics = {
    'AP': 100 * dino_eval[0], 'AP50': 100 * dino_eval[1], 'AP75': 100 * dino_eval[2],
    'AP_small': 100 * dino_eval[3], 'AP_medium': 100 * dino_eval[4], 'AP_large': 100 * dino_eval[5],
}
(DINO_OUT / 'evaluation.json').write_text(json.dumps(dino_metrics, indent=2))
print(json.dumps(dino_metrics, indent=2))

In [ ]:
# Qualitative DINO detections using the fine-tuned checkpoint and the repository's postprocessor.
sys.path.insert(0, str(PROJECT / 'DINO'))
from util.slconfig import SLConfig
from main import build_model_main
import datasets.transforms as T
from PIL import Image, ImageDraw, ImageFont

dino_args = SLConfig.fromfile(str(PROJECT / 'DINO' / 'config' / 'DINO' / 'DINO_4scale.py'))
dino_args.device = 'cuda'
dino_args.num_classes = 20
dino_args.dn_labelbook_size = 21
dino_model, _, postprocessors = build_model_main(dino_args)
checkpoint_path = DINO_OUT / 'checkpoint_best_regular.pth'
if not checkpoint_path.exists(): checkpoint_path = DINO_OUT / 'checkpoint.pth'
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
dino_model.load_state_dict(checkpoint['model'])
dino_model.cuda().eval()
transform = T.Compose([
    T.RandomResize([800], max_size=1333),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_names = (VOC_ROOT / 'ImageSets' / 'Main' / 'val.txt').read_text().split()
sample_names = random.Random(SEED).sample(val_names, 4)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, stem in zip(axes.ravel(), sample_names):
    image = Image.open(VOC_ROOT / 'JPEGImages' / f'{stem}.jpg').convert('RGB')
    tensor, _ = transform(image, None)
    with torch.no_grad():
        raw = dino_model(tensor[None].cuda())
    result = postprocessors['bbox'](raw, torch.tensor([[image.height, image.width]], device='cuda'))[0]
    keep = result['scores'] >= SCORE_THRESHOLD
    draw = ImageDraw.Draw(image)
    for score, label, box in zip(result['scores'][keep].cpu(), result['labels'][keep].cpu(), result['boxes'][keep].cpu()):
        x0, y0, x1, y1 = box.tolist()
        draw.rectangle((x0, y0, x1, y1), outline='red', width=3)
        class_name = VOC_CLASSES[int(label)] if int(label) < len(VOC_CLASSES) else f'class_{int(label)}'
        draw.text((x0 + 2, max(0, y0 - 14)), f'{class_name} {float(score):.2f}', fill='red')
    ax.imshow(image)
    ax.set_title(f'{stem}.jpg')
    ax.axis('off')
fig.suptitle('DINO-4scale detections on VOC 2012 validation images', fontsize=16)
fig.tight_layout()
fig.savefig(DINO_OUT / 'qualitative_detections.png', dpi=180, bbox_inches='tight')
plt.show()

## 7. Quantitative comparison and analysis

The two AP50 values are computed on the same VOC 2012 validation images. Detectron2 reports the VOC AP50 convention, while DINO's COCO evaluator reports AP at IoU 0.50; this makes AP50 the closest direct comparison. Overall COCO AP for DINO averages IoU thresholds from 0.50 to 0.95 and is therefore stricter.

In [ ]:
frcnn_saved = json.loads((FRCNN_OUT / 'evaluation.json').read_text())
dino_saved = json.loads((DINO_OUT / 'evaluation.json').read_text())
frcnn_ap50 = float(frcnn_saved['bbox']['AP50'])
dino_ap50 = float(dino_saved['AP50'])
comparison = pd.DataFrame([
    {'Model': 'Faster R-CNN R101-FPN 3x', 'Initialization': 'COCO', 'Fine-tuning budget': f'{FRCNN_MAX_ITER} iterations', 'Validation AP50': frcnn_ap50},
    {'Model': 'DINO-4scale R50', 'Initialization': 'COCO', 'Fine-tuning budget': f'{DINO_EPOCHS} epoch(s)', 'Validation AP50': dino_ap50},
])
display(comparison.style.format({'Validation AP50': '{:.2f}'}))

winner = comparison.loc[comparison['Validation AP50'].idxmax(), 'Model']
gap = abs(frcnn_ap50 - dino_ap50)
analysis = f'''
### Results analysis

Faster R-CNN reached **{frcnn_ap50:.2f} AP50**, while DINO reached **{dino_ap50:.2f} AP50** on the VOC 2012 validation split. Under these limited fine-tuning budgets, **{winner}** was higher by **{gap:.2f} AP50 points**. This comparison reflects both architecture and optimization-budget differences; it is not evidence that one architecture is universally superior.

The Faster R-CNN loss plot above should be inspected for a sustained downward trend and late-training stability. A loss that is still falling steeply at the final iteration suggests under-training, while a flat or noisy tail suggests that more iterations at the same learning rate may offer limited benefit. The qualitative panels should be checked for missed small objects, duplicate boxes, localization errors, and confusions among visually similar VOC classes.

DINO was fine-tuned for only {DINO_EPOCHS} epoch(s), so its result is especially sensitive to learning rate, batch size, and random seed. A fairer follow-up would hold the data split fixed, run two or more seeds, and compare 1 versus 2 epochs or learning rates 1e-5 versus 5e-6.
'''
display(__import__('IPython').display.Markdown(analysis))
(OUTPUT_ROOT / 'results_analysis.md').write_text(analysis + '\n' + comparison.to_markdown(index=False))